# Notebook for preprocessing data

In [20]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


In [21]:
df = pd.read_csv("../raw_data/articles.csv")
df.drop(columns=["detail_desc"]).head(5)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear"
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear"


In [22]:
print(df['product_group_name'].unique())
product_group = 'Shoes'

<StringArray>
[   'Garment Upper body',             'Underwear',        'Socks & Tights',
    'Garment Lower body',           'Accessories',                 'Items',
             'Nightwear',               'Unknown',   'Underwear/nightwear',
                 'Shoes',              'Swimwear',     'Garment Full body',
              'Cosmetic',      'Interior textile',                  'Bags',
             'Furniture', 'Garment and Shoe care',                   'Fun',
            'Stationery']
Length: 19, dtype: str


In [23]:
print(f"The subcategories of {product_group} are: ")
df[df['product_group_name']==product_group]['product_type_name'].unique()

The subcategories of Shoes are: 


<StringArray>
[         'Boots',       'Sneakers',     'Other shoe',        'Sandals',
       'Slippers',     'Ballerinas',      'Flat shoe',          'Wedge',
          'Pumps',      'Flip flop',         'Bootie', 'Heeled sandals',
     'Flat shoes',          'Heels',      'Moccasins',    'Pre-walkers']
Length: 16, dtype: str

In [24]:
df_filtered = df[df['product_group_name']==product_group]
df_filtered.head(3)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
124,181160009,181160,Eva chelsea boot,87,Boots,Shoes,1010016,Solid,17,Yellowish Brown,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Chelsea boots with elasticated gores in the si...
257,212042036,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,1,Other,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...
258,212042043,212042,Mimmi sneaker,94,Sneakers,Shoes,1010016,Solid,9,Black,...,Divided Shoes,D,Divided,2,Divided,52,Divided Accessories,1020,Shoes,Cotton trainers with closed lacing and a loop ...


In [25]:
df_filtered['index_group_name'].unique()

<StringArray>
['Divided', 'Menswear', 'Ladieswear', 'Baby/Children', 'Sport']
Length: 5, dtype: str

In [26]:
df_trans = pd.read_csv("../raw_data/transactions_train.csv")
df_trans['t_dat']= pd.to_datetime(df_trans['t_dat'])
df_trans.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [27]:
df_trans_filtered = df_trans[df_trans['article_id'].isin(df_filtered['article_id'])]

In [28]:
df_trans_filtered.shape

(745521, 5)

In [29]:
print("Saving filtered dataFrames")

Saving filtered dataFrames


In [30]:
df_filtered.to_csv("../raw_data/articles_filtered.csv")

In [31]:
df_trans_filtered.to_csv("../raw_data/transactions_filtered.csv")

## Filter images

In [32]:
import os
import shutil
from pathlib import Path

In [33]:
needed_article_ids = df_filtered['article_id'].unique()
needed_article_ids.shape

(5283,)

In [34]:
source_images_path = Path('../raw_data/images_256_256')
dest_images_path = Path('../raw_data/images_filtered')

In [35]:
copied_count = 0
missing_count = 0

for article_id in needed_article_ids:
    article_str = str(article_id).zfill(10)
    subfolder = article_str[:3]

    source_subfolder = source_images_path / subfolder
    dest_subfolder = dest_images_path / subfolder
    source_file = source_subfolder / f"{article_str}.jpg"
    dest_file = dest_subfolder / f"{article_str}.jpg"

    if source_file.exists():
        dest_subfolder.mkdir(exist_ok=True)

        shutil.copy2(source_file, dest_file)
        copied_count += 1
    else:
        missing_count += 1
        df_filtered.drop(df_filtered[df_filtered['article_id'] == article_id].index, inplace=True)
        df_trans_filtered.drop(df_trans_filtered[df_trans_filtered['article_id'] == article_id].index, inplace=True)

print(f"Copied {copied_count} images")
print(f"Missing images: {missing_count}")
print(f"Total article IDs: {needed_article_ids.shape}")

Copied 5156 images
Missing images: 127
Total article IDs: (5283,)


In [36]:
df_filtered.shape

(5156, 25)

In [37]:
df_trans_filtered.shape

(738255, 5)

In [38]:
df_filtered.to_csv("../raw_data/articles_filtered.csv")
df_trans_filtered.to_csv("../raw_data/transactions_filtered.csv")